In [ ]:
# Example 1: inspect the public API surface.
import pyperclip

public_names = [n for n in dir(pyperclip) if not n.startswith('_')]
print(f"Number of public names: {len(public_names)}")
print(f"Sample: {public_names[:15]}")

# The canonical primary API:
for fn_name in ('copy', 'paste', 'determine_clipboard', 'set_clipboard'):
    assert hasattr(pyperclip, fn_name), f"missing pyperclip.{fn_name}"
    assert callable(getattr(pyperclip, fn_name)), f"pyperclip.{fn_name} not callable"
print("copy / paste / determine_clipboard / set_clipboard are all callable")


In [ ]:
# Example 2: exception class is present and is an Exception subclass.
import pyperclip

assert hasattr(pyperclip, 'PyperclipException'), "missing PyperclipException"
assert issubclass(pyperclip.PyperclipException, Exception)

# Construct + raise/catch round-trip.
try:
    raise pyperclip.PyperclipException("test error")
except pyperclip.PyperclipException as e:
    caught = str(e)

assert caught == "test error"
print(f"PyperclipException round-trip OK: {caught}")


In [ ]:
# Example 3: set_clipboard accepts known backend names; unknown names raise.
import pyperclip

# Known backends per pyperclip source: 'pbcopy', 'pbobjc', 'qt', 'xclip',
# 'xsel', 'klipper', 'windows', 'no', plus 'wsl' on some versions.
# 'no' is the safe always-available no-op backend.
pyperclip.set_clipboard('no')
print("set_clipboard('no') accepted")

raised = False
try:
    pyperclip.set_clipboard('definitely-not-a-real-backend')
except ValueError as e:
    assert "Argument must be one of" in str(e), f"unexpected ValueError message: {e}"
    raised = True
    print(f"Unknown backend raised: {type(e).__name__}: {e}")

assert raised, "expected set_clipboard with garbage name to raise"


In [ ]:
# Example 4: the 'no' backend is the explicit "no clipboard" backend — its
# copy()/paste() raise PyperclipException by design (there is no mechanism).
import pyperclip

pyperclip.set_clipboard('no')

for op, call in (('copy', lambda: pyperclip.copy('hello world')),
                 ('paste', lambda: pyperclip.paste())):
    raised = False
    try:
        call()
    except pyperclip.PyperclipException as e:
        raised = True
        print(f"{op}() with 'no' backend correctly raised: {type(e).__name__}")
    assert raised, f"expected {op}() on the 'no' backend to raise PyperclipException"

In [ ]:
# Example 5: determine_clipboard returns a (copy, paste) tuple of callables.
import pyperclip

copy_fn, paste_fn = pyperclip.determine_clipboard()
print(f"copy_fn = {copy_fn}")
print(f"paste_fn = {paste_fn}")

assert callable(copy_fn), "copy_fn must be callable"
assert callable(paste_fn), "paste_fn must be callable"

# lazy_load_stub_copy / lazy_load_stub_paste are the fallback when no real
# backend is found. Either real or stub callables are valid here.
print(f"copy_fn.__name__ = {getattr(copy_fn, '__name__', '?')}")
print(f"paste_fn.__name__ = {getattr(paste_fn, '__name__', '?')}")

assert hasattr(pyperclip, 'is_available')
available = pyperclip.is_available()
print(f"is_available() = {available}")
assert isinstance(available, bool)